<a href="https://colab.research.google.com/github/jorshimayor/AI-Engineering-Flow_Research/blob/main/jupyter/week4_calibration_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 4 Protocol: Difficulty Calibration (Dev Set Only)

<a href="https://colab.research.google.com/github/jorshimayor/AI-Engineering-Flow_Research/blob/main/jupyter/week4_calibration_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

The Week 3 problems were too easy at 1.5B and 3B (Act-only 97% to 100%), so protocol v1.1 uses a harder generator ([`src/react_tasks_v2.py`](../src/react_tasks_v2.py)) with three difficulty levels. This notebook picks the level, using the rule written in the protocol **before** this run (`notes/week_4/eval_protocol.md`, section 3.2.1):

1. Act-only on the 40 **dev** problems at levels 1, 2 and 3, for Qwen2.5-1.5B and 3B.
2. Eligible levels: Act-only between 40% and 85% for both models. Pick the one closest to 62.5%.
3. Direct and ReAct on dev at the chosen level, to check the other validity gates.

It only ever touches the **dev** split. The 400 test problems stay unseen until the protocol is frozen.

**Before you start:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`. Roughly 60 to 90 minutes on a free T4 (an estimate: the harder problems are longer than Week 3's). Results are saved one problem at a time, so a disconnect only costs the problem in progress.

## 1. Check the GPU

In [2]:
import torch
assert torch.cuda.is_available(), "No GPU: Runtime → Change runtime type → T4 GPU, then run again."
print(torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB")

Tesla T4 16 GB


## 2. Get the code

In [3]:
REPO = "https://github.com/jorshimayor/AI-Engineering-Flow_Research.git"
BRANCH = "main"
import os
if not os.path.exists("AI-Engineering-Flow_Research"):
    !git clone --depth 1 --branch {BRANCH} {REPO}
%cd AI-Engineering-Flow_Research
!pip install -q "transformers==4.49.0" accelerate
!git log --oneline -1

Cloning into 'AI-Engineering-Flow_Research'...
remote: Enumerating objects: 49, done.
remote: Counting objects: 100% (49/49), done.
remote: Compressing objects: 100% (45/45), done.
remote: Total 49 (delta 2), reused 42 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (49/49), 2.14 MiB | 26.38 MiB/s, done.
Resolving deltas: 100% (2/2), done.
/content/AI-Engineering-Flow_Research
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 110.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 111.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
g

## 3. Keep results on Google Drive (recommended)

In [4]:
USE_DRIVE = True
RESULTS = "results/week4_dev"
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    target = "/content/drive/MyDrive/AI-Engineering/week4_dev"
    os.makedirs(target, exist_ok=True); os.makedirs("results", exist_ok=True)
    if not os.path.islink(RESULTS):
        if os.path.isdir(RESULTS):
            os.rmdir(RESULTS)
        os.symlink(target, RESULTS)
os.makedirs(RESULTS, exist_ok=True)
print("results go to:", os.path.realpath(RESULTS))

Mounted at /content/drive
results go to: /content/drive/MyDrive/AI-Engineering/week4_dev


## 4. Check the generator

Automatic checks on every split and level: answers equal their solution expressions, answers are positive integers, the step count matches the level, and no test template shares an exemplar's operations (gate 4). Test questions are checked but never printed.

In [5]:
import sys, json, gc
from pathlib import Path
sys.path.insert(0, "src")
from react_tasks_v2 import make_problems, check_problems, leakage_check
for split in ("dev", "test"):
    for L in (1, 2, 3):
        c = check_problems(make_problems(split, L))
        print(f"{split:4} level {L}: {c['n']:3} problems, errors {len(c['errors'])}")
        assert not c["errors"], c["errors"][:3]
issues = leakage_check()
print("leakage:", issues or "none"); assert not issues
print("\nOne dev problem per level:")
for L in (1, 2, 3):
    p = make_problems("dev", L)[0]; print(f"[L{L}] {p['question']}  -> {p['gold']:,}")

dev  level 1:  40 problems, errors 0
dev  level 2:  40 problems, errors 0
dev  level 3:  40 problems, errors 0
test level 1: 400 problems, errors 0
test level 2: 400 problems, errors 0
test level 3: 400 problems, errors 0
leakage: none

One dev problem per level:
[L1] A bakery bakes 45 trays of rolls every day. Each tray holds 36 rolls. Over 7 days the bakery sells 3,702 rolls in total. The bakery employs 16 bakers. How many rolls are left unsold after those days?  -> 7,638
[L2] Over 13 days the bakery sells 6,329 rolls in total. The bakery employs 14 bakers. Each tray holds 36 rolls. A bakery bakes 44 trays of rolls every day. How many rolls are left unsold after those days?  -> 14,263
[L3] Each tray holds 23 rolls. The bakery opened 28 years ago. Over 7 days the bakery sells 15,331 rolls in total. A bakery bakes 41 trays of rolls per hour, for 5 hours every day. Each roll sells for $4. How many rolls are left unsold after those days?  -> 17,674


## 5. Stage A: Act-only on dev at all three levels

Each model is loaded once and freed before the next.

In [ ]:
from react_agent import LocalLLM
from run_react_bench import run, failure_type, is_format_failure
from stats import wilson_ci
import torch

MODELS = ["Qwen/Qwen2.5-1.5B-Instruct", "Qwen/Qwen2.5-3B-Instruct"]
LEVELS = [1, 2, 3]
MAX_STEPS = 8
def out_path(model, level): return Path(RESULTS) / f"{model.split('/')[-1]}-L{level}.jsonl"

for model in MODELS:
    llm = LocalLLM(model)
    for L in LEVELS:
        run(model, ["act"], make_problems("dev", L), out_path(model, L), MAX_STEPS, llm=llm)
    del llm; gc.collect(); torch.cuda.empty_cache()

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/3.97G [00:00<?, ?B/s]

## 6. Apply the selection rule (protocol 3.2.1)

In [ ]:
def load(model, level):
    f = out_path(model, level)
    return [json.loads(l) for l in f.open()] if f.exists() else []

def acc(rows, mode):
    rs = [r for r in rows if r["mode"] == mode]
    return (sum(r["correct"] for r in rs) / len(rs), len(rs)) if rs else (float("nan"), 0)

table = {}
print(f"{'level':6}" + "".join(f"{m.split('-')[1]:>22}" for m in MODELS) + "   eligible?")
for L in LEVELS:
    accs = [acc(load(m, L), "act") for m in MODELS]
    ok = all(0.40 <= a <= 0.85 and n == 40 for a, n in accs)
    table[L] = dict(act={m: a for m, (a, _) in zip(MODELS, accs)}, eligible=ok)
    cells = "".join(f"{a:8.1%} (CI {wilson_ci(round(a*n), n)[0]:.0%} to {wilson_ci(round(a*n), n)[1]:.0%})" for a, n in accs)
    print(f"{L:<6}{cells}   {'yes' if ok else 'no'}")

eligible = [L for L in LEVELS if table[L]["eligible"]]
if eligible:
    CHOSEN = min(eligible, key=lambda L: abs(sum(table[L]["act"].values()) / len(MODELS) - 0.625))
    print(f"\nChosen level: {CHOSEN}  (eligible: {eligible})")
else:
    CHOSEN = None
    print("\nNo level is eligible: gate 5 fails. Share these results; the generator needs a change on dev before freezing.")

## 7. Stage B: Direct and ReAct on dev at the chosen level

Checks gate 1 (direct ≤ 10%) and gate 3 (format failures ≤ 20% in both modes) before freezing.

In [ ]:
if CHOSEN is not None:
    for model in MODELS:
        llm = LocalLLM(model)
        run(model, ["direct", "react"], make_problems("dev", CHOSEN), out_path(model, CHOSEN), MAX_STEPS, llm=llm)
        del llm; gc.collect(); torch.cuda.empty_cache()

## 8. Gate report and calibration record

In [ ]:
from collections import Counter
report = dict(protocol_version="1.1", split="dev", levels=LEVELS, chosen_level=CHOSEN, selection_table=table, gates={})
if CHOSEN is not None:
    for model in MODELS:
        rows = load(model, CHOSEN)
        a_act, _ = acc(rows, "act"); a_react, _ = acc(rows, "react"); a_dir, _ = acc(rows, "direct")
        ff = {m: sum(is_format_failure(r) for r in rows if r["mode"] == m) / max(1, sum(r["mode"] == m for r in rows)) for m in ("act", "react")}
        gates = {"1_direct_le_10pct": a_dir <= 0.10, "3_format_fail_le_20pct": all(v <= 0.20 for v in ff.values()),
                 "5_act_in_40_85": 0.40 <= a_act <= 0.85}
        report["gates"][model] = dict(direct=a_dir, act=a_act, react=a_react, format_fail=ff, passed=gates)
        print(f"\n{model}: direct {a_dir:.0%} | Act {a_act:.0%} | ReAct {a_react:.0%} | format failures Act {ff['act']:.0%}, ReAct {ff['react']:.0%}")
        for g, ok in gates.items(): print(f"   gate {g}: {'PASS' if ok else 'FAIL'}")
        print("   ReAct failure types:", dict(Counter(failure_type(r) for r in rows if r['mode'] == 'react' and not r['correct'])))
    report["all_gates_passed"] = all(all(v["passed"].values()) for v in report["gates"].values())
    print("\nALL GATES PASSED" if report["all_gates_passed"] else "\nSome gates failed: share the results before freezing.")
json.dump(report, open(Path(RESULTS) / "calibration.json", "w"), indent=1, default=str)
print("saved", Path(RESULTS) / "calibration.json")

## 9. Save the results

In [ ]:
!cd results && zip -qr /content/week4_dev_calibration.zip week4_dev/
from google.colab import files
files.download("/content/week4_dev_calibration.zip")

Send the zip back (or unzip it into `results/week4_dev/` in the repo). Next: record the chosen level in the protocol, freeze it with a commit, then run the 400 test problems.